# Phân tích dữ liệu sản phẩm — Folder `phân tích dữ liệu`

Notebook này đọc tất cả file `*.json` trong thư mục `phân tích dữ liệu`,
phân tích schema, thống kê số lượng, brand, flavor, giá và trực quan hóa.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
%matplotlib inline

## 1. Đọc folder và liệt kê file

In [ ]:
DATA_DIR = Path(r'C:\Users\Admin\profit_new\profit\phân tích dữ liệu')
print('Thư mục:', DATA_DIR)
print()
json_files = sorted(DATA_DIR.glob('*.json'))
print(f'Tìm thấy {len(json_files)} file JSON:')
for f in json_files:
    size_kb = f.stat().st_size / 1024
    print(f'  - {f.name:35s} {size_kb:8.1f} KB')

## 2. Load tất cả file JSON vào dict

In [ ]:
datasets = {}
for f in json_files:
    with open(f, 'r', encoding='utf-8') as fp:
        datasets[f.stem] = json.load(fp)

for name, data in datasets.items():
    print(f'{name:30s} -> {len(data):4d} records')

## 3. Tổng quan số records mỗi dataset

In [ ]:
summary = pd.DataFrame([
    {'dataset': name, 'records': len(data)}
    for name, data in datasets.items()
]).sort_values('records', ascending=False)
summary

## 4. Phân tích schema — cấu trúc mỗi record

In [ ]:
def describe_schema(records):
    if not records: return
    print('Top-level keys:', list(records[0].keys()))
    rec = records[0]
    if 'metadata' in rec:
        print('  metadata keys:', list(rec['metadata'].keys()))
    if 'attributes' in rec:
        print('  attributes keys:', list(rec['attributes'].keys()))
    print()

for name, data in datasets.items():
    print(f'=== {name} ===')
    describe_schema(data)

## 5. Liệt kê tất cả trường `metadata` và `attributes` xuất hiện

In [ ]:
all_meta_keys, all_attr_keys = set(), set()
for data in datasets.values():
    for rec in data:
        if 'metadata' in rec:
            all_meta_keys.update(rec['metadata'].keys())
        if 'attributes' in rec:
            all_attr_keys.update(rec['attributes'].keys())

print('Các trường metadata chung:')
for k in sorted(all_meta_keys):
    print(f'  - {k}')
print()
print('Các trường attributes (riêng theo category):')
for k in sorted(all_attr_keys):
    print(f'  - {k}')

## 6. Thống kê theo từng dataset (brand / flavor / giá)

In [ ]:
def stat_dataset(name, data):
    rows = []
    for rec in data:
        m = rec.get('metadata', {})
        rows.append({
            'id': rec.get('id'),
            'name': m.get('name'),
            'brand': m.get('brand'),
            'category': m.get('category'),
            'flavor': m.get('flavor'),
            'price': m.get('price'),
            'origin': m.get('origin_country'),
        })
    df = pd.DataFrame(rows)
    print(f'=== {name} ({len(df)} records) ===')
    if 'category' in df.columns:
        print(f'Category: {sorted(df.category.dropna().unique().tolist())}')
    if 'brand' in df.columns:
        print(f'Brands ({df.brand.nunique()}): {sorted(df.brand.dropna().unique())}')
    if 'flavor' in df.columns:
        print(f'Flavors ({df.flavor.nunique()}): {sorted(df.flavor.dropna().unique())}')
    if 'origin' in df.columns:
        print(f'Origin: {sorted(df.origin.dropna().unique())}')
    if 'price' in df.columns and df.price.notna().any():
        print(f'Price: min={df.price.min():.2f}  max={df.price.max():.2f}  mean={df.price.mean():.2f}')
    print()
    return df

dfs = {}
for name, data in datasets.items():
    dfs[name] = stat_dataset(name, data)

## 7. Gộp tất cả thành 1 bảng lớn

In [ ]:
all_df = pd.concat(
    [df.assign(dataset=name) for name, df in dfs.items() if not df.empty],
    ignore_index=True,
)
print(f'Tổng: {len(all_df)} records')
all_df.head(10)

## 8. Đếm số sản phẩm theo brand (top 15)

In [ ]:
top_brands = (
    all_df.dropna(subset=['brand'])
    .groupby(['dataset', 'brand'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(15)
)
top_brands

## 9. Trực quan hóa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

summary.plot.bar(x='dataset', y='records', ax=axes[0],
                 legend=False, color='steelblue', title='Số records mỗi file')
axes[0].set_ylabel('Records')
axes[0].tick_params(axis='x', rotation=30)

all_df['dataset'].value_counts().plot.pie(
    ax=axes[1], autopct='%1.1f%%', startangle=90,
    title='Tỉ lệ records theo dataset')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
all_df.boxplot(column='price', by='dataset', ax=ax)
ax.set_title('Phân bố giá theo dataset')
ax.set_xlabel('Dataset')
ax.set_ylabel('Price (USD)')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
top15 = (
    all_df['brand'].dropna().value_counts().head(15).sort_values()
)
top15.plot.barh(figsize=(10, 6), color='teal')
plt.title('Top 15 brands (nhiều sản phẩm nhất)')
plt.xlabel('Số sản phẩm')
plt.tight_layout()
plt.show()

## 10. Kết luận

In [ ]:
print('Tổng kết:')
print(f'  - Số file JSON: {len(datasets)}')
print(f'  - Tổng records: {sum(len(d) for d in datasets.values())}')
print(f'  - Số brand unique: {all_df.brand.nunique()}')
print(f'  - Số flavor unique: {all_df.flavor.nunique()}')
if 'price' in all_df.columns and all_df.price.notna().any():
    print(f'  - Khoảng giá: {all_df.price.min():.2f} -> {all_df.price.max():.2f} USD')